# 01 Intro Pipeline

This notebook introduces the WiCyS campus SOC workflow.

It covers:

- seed data inspection
- ingested event review
- bounded triage actions
- governance-oriented interpretation


## Goals

By the end of this notebook, you should be able to:

1. inspect the seed datasets
2. understand the event structure
3. review ingested detector output
4. connect the pipeline to governance decisions


In [ ]:
from pathlib import Path
import pandas as pd
import json

DATA_DIR = Path("/workspace/data")
print("Data directory:", DATA_DIR)
print("Files:", [p.name for p in DATA_DIR.glob("*")])


In [ ]:
seed_lms = pd.read_csv(DATA_DIR / "seed_lms_events.csv") if (DATA_DIR / "seed_lms_events.csv").exists() else pd.DataFrame()
seed_email = pd.read_csv(DATA_DIR / "seed_email_events.csv") if (DATA_DIR / "seed_email_events.csv").exists() else pd.DataFrame()

print("LMS rows:", len(seed_lms))
print("Email rows:", len(seed_email))
seed_lms.head(), seed_email.head()


In [ ]:
ingested_path = DATA_DIR / "ingested_events.jsonl"
rows = []
if ingested_path.exists():
    with open(ingested_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

print("Ingested events:", len(rows))
rows[:2]


In [ ]:
actions = []
for r in rows:
    dr = r.get("detector_result", {})
    if dr.get("action"):
        actions.append(dr["action"])

pd.Series(actions).value_counts(dropna=False)


## Bounded triage

The lab supports only three permitted triage outputs:

- `allow`
- `queue_for_review`
- `escalate`

This bounded action space is a governance feature, not just a UI choice.
